# Practice: Turn Counts Into a Paragraph You Can Defend

**Computational Text Analysis Pack · Module 11 — From Counts to Argument**

*Technique:* Drafting the methods and findings text; hedging that is accurate rather than vague

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/10-visualising-and-arguing/11d_practice_writing_it_up.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **You have a result. What exactly can you write?**

This notebook produces a finding and then drafts three versions of the paragraph
reporting it — one overclaiming, one so hedged it says nothing, and one you could
defend. Writing is the last method in the pipeline and the only one whose output
anyone reads.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
import numpy as np
# A concrete finding, computed here so the prose has something to be about.
A, B = "health", "politics"
ca, cb = Counter(), Counter()
na_docs = df[df["category"] == A]["text"].tolist()
nb_docs = df[df["category"] == B]["text"].tolist()
for t in na_docs: ca.update(tokenise(str(t)))
for t in nb_docs: cb.update(tokenise(str(t)))
na, nb = sum(ca.values()), sum(cb.values())

WORD = "community"           # <- change this
rate_a = ca[WORD]/na*1e6
rate_b = cb[WORD]/nb*1e6
lr = np.log2((ca[WORD]+.5)/na) - np.log2((cb[WORD]+.5)/nb)
docs_a = sum(1 for t in na_docs if WORD in tokenise(str(t)))
docs_b = sum(1 for t in nb_docs if WORD in tokenise(str(t)))

print(f"word: '{WORD}'")
print(f"  {A:<10} {ca[WORD]:>6} occurrences, {rate_a:>8.1f} per million, "
      f"in {docs_a}/{len(na_docs)} documents")
print(f"  {B:<10} {cb[WORD]:>6} occurrences, {rate_b:>8.1f} per million, "
      f"in {docs_b}/{len(nb_docs)} documents")
print(f"  log ratio: {lr:+.2f}")

## Draft 1 — the overclaim

> *African health journalism is fundamentally community-oriented, in contrast to
> political reporting, which centres the state.*

Everything wrong with this is instructive:

- **"African health journalism"** — the corpus is 16 languages of online news
  with a per-language cap. It is not African journalism.
- **"fundamentally"** — a frequency ratio does not license a claim about
  essential character.
- **"in contrast to... centres the state"** — we measured one word. The second
  clause was never tested.

## Draft 2 — the useless hedge

> *There may possibly be some indication that the word "community" could
> potentially occur with differing frequency across categories in this corpus,
> although further research would be needed.*

This is not caution, it is abdication. It reports no number, commits to nothing,
and gives the reader nothing to disagree with. Hedging is supposed to make a
claim *precise*, not evacuate it.

In [ ]:
# Draft 3 — assembled from what we actually measured.
draft = f'''In this corpus, "{WORD}" occurs at {rate_a:.0f} per million words in
{A} articles against {rate_b:.0f} in {B} articles (log ratio {lr:+.2f}), and
appears in {docs_a} of {len(na_docs)} {A} documents against {docs_b} of
{len(nb_docs)} {B} documents. The difference is therefore not driven by a small
number of repetitive articles. The corpus is capped per language and drawn from
online news outlets, so this is a claim about the sampled outlets rather than
about African journalism generally.'''
print(" ".join(draft.split()))
print()
print("Count what that paragraph does: gives both rates, the effect size, the")
print("document spread (so a reader knows it is not three articles), and the")
print("limit of the corpus. Every clause is checkable against a number above.")

In [ ]:
# The sentences that make a limits section, generated from the data itself.
print("LIMITS, stated from what we know about the corpus:\n")
print(f"- The corpus contains {len(df):,} documents across "
      f"{df['language'].nunique()} languages, capped per language, so language")
print("  proportions are an artefact of collection rather than of publication.")
print(f"- Category labels come with the dataset and were not verified here;")
print(f"  module 17's agreement statistics were not run on them.")
print("- Findings are about the sampled outlets. MasakhaNEWS draws heavily on")
print("  one large broadcaster, so 'African news' would be the wrong subject.")
print("- No date information is attached, so no claim about change over time is")
print("  available from this corpus.")
print()
print("Write these before the findings, not after. A limits section drafted")
print("first tells you which findings you are allowed to look for.")

## The three tests before you submit

1. **Could a reader disagree with this sentence using your own numbers?** If
   not, it is either unfalsifiable or unsupported.
2. **Does the subject of your sentence match the subject of your corpus?** This
   is where most overclaiming happens: "African journalism" for a capped sample
   of online news.
3. **Is every hedge doing work?** "Approximately", with a figure, is useful.
   "Perhaps to some extent" is not a hedge, it is a retreat.

### 🔧 Your turn 1

Take a finding from your own work and write all three drafts. Show the third to someone who has not seen the data and ask what they think you measured.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The common discovery is that a reader takes the subject of your sentence far
more broadly than you intended — they hear "African news" where you meant "1,400
sampled articles from four outlets". That gap between what you wrote and what
was heard is the thing to fix, and it is almost always fixed by naming the
corpus in the sentence rather than in a footnote. This is the same discipline
module 22c's absence audit asks for, arriving at the level of the sentence.

</details>